# `stream_distance`: per-row step precompute benchmark

`pyflwdir.streams.stream_distance` computes the distance from every cell to its outlet
(or to the next masked cell) by walking the flow network from downstream to upstream.

The previous implementation called `gis_utils.distance(idx0, idx_ds, ...)` for **every
cell**. For geographic coordinates (`latlon=True`) that function re-evaluates the
metre-per-degree factors `degree_metres_y` / `degree_metres_x` (several `cos()` calls
each) for every single cell, even though the factor only depends on the **row**
(latitude). For a large basin this means hundreds of millions of redundant
trigonometric calls.

The optimized implementation precomputes the step length to the 8 neighboring cells
**once per row** (5 numbers per row: horizontal, down, down-diagonal, up,
up-diagonal) and looks it up inside the loop. Non-neighbor jumps (e.g. `nextxy` flow
directions) still fall back to the generic `gis_utils.distance`.

This notebook verifies that the new implementation is bit-identical to the previous
one and benchmarks the speedup on synthetic D8 data and on the built-in Rhine basin.

## How to run

From the repository root:

```bash
cd examples
python -m jupyter nbconvert --to notebook --execute --inplace stream_distance_benchmark.ipynb
```

or open it interactively with `jupyter notebook stream_distance_benchmark.ipynb`.

This change is intentionally **bit-identical** to the current pyflwdir
implementation.


In [1]:
import time
import numpy as np
from numba import njit

import pyflwdir
from pyflwdir import streams, gis_utils

print(f"pyflwdir {pyflwdir.__version__} at {pyflwdir.__file__}")

pyflwdir 0.5.13.dev0 at /Users/jianglulu/GitHub/pyflwdir/pyflwdir/__init__.py


In [2]:
@njit
def stream_distance_real_old(idxs_ds, seq, ncol, mask, latlon, transform):
    """Previous implementation: one gis_utils.distance call per cell."""
    dist = np.full(idxs_ds.size, -9999.0, dtype=np.float32)
    dist[seq] = 0
    for idx0 in seq:  # down- to upstream
        idx_ds = idxs_ds[idx0]
        # sum distances; skip if at pit or mask is True
        if idx0 == idx_ds or (mask is not None and mask[idx0]):
            continue
        d = gis_utils.distance(idx0, idx_ds, ncol, latlon, transform)
        dist[idx0] = dist[idx_ds] + d
    return dist

In [3]:
def synthetic_d8(ncol, nrow):
    """Diagonal-heavy acyclic D8 raster.

    Interior cells flow up-left, column 0 flows up, row 0 flows left and
    (0, 0) is the pit. Every cell points to a smaller linear index, so
    ``np.arange(size)`` is a valid down-to-upstream sequence.
    """
    size = ncol * nrow
    idx0 = np.arange(size, dtype=np.int64)
    r = idx0 // ncol
    c = idx0 % ncol
    idxs_ds = idx0.copy()
    m = (r > 0) & (c > 0)
    idxs_ds[m] = idx0[m] - ncol - 1  # diagonal up-left
    m = (r > 0) & (c == 0)
    idxs_ds[m] = idx0[m] - ncol      # up
    m = (r == 0) & (c > 0)
    idxs_ds[m] = idx0[m] - 1         # left
    seq = np.arange(size, dtype=np.int64)
    return idxs_ds, seq

In [4]:
# 1 arc-second geographic grid and a 30 x 10 m projected grid
transform_ll = np.array([1 / 3600, 0, 0, 0, -1 / 3600, 5.0])
transform_xy = np.array([30.0, 0, 0, 0, -10.0, 100.0])

ncol = nrow = 200
idxs_ds, seq = synthetic_d8(ncol, nrow)

# bit-identical results for all supported index dtypes and both CRS flavours
for name, idxs_ds_, seq_ in [
    ("int64", idxs_ds, seq),
    ("int32", idxs_ds.astype(np.int32), seq.astype(np.int32)),
    ("uint32", idxs_ds.astype(np.uint32), seq.astype(np.uint32)),
]:
    for latlon, transform in [(True, transform_ll), (False, transform_xy)]:
        old = stream_distance_real_old(idxs_ds_, seq_, ncol, None, latlon, transform)
        new = streams.stream_distance(idxs_ds_, seq_, ncol, latlon=latlon, transform=transform)
        assert np.array_equal(new, old), (name, latlon)
        print(f"{name:>6}  latlon={latlon}: bit-identical")

# nextxy-style non-neighbor jumps take the generic fallback and still match
idxs_ds_j = idxs_ds.copy()
idxs_ds_j[100] = 90   # 10 cells left, same row
idxs_ds_j[101] = 51   # one row up
old = stream_distance_real_old(idxs_ds_j, seq, ncol, None, True, transform_ll)
new = streams.stream_distance(idxs_ds_j, seq, ncol, latlon=True, transform=transform_ll)
assert np.array_equal(new, old)
print("nextxy-jump  latlon=True : bit-identical")

# mask behaviour is unchanged
mask = np.zeros(idxs_ds.size, dtype=bool)
mask[50] = True
old = stream_distance_real_old(idxs_ds, seq, ncol, mask, True, transform_ll)
new = streams.stream_distance(idxs_ds, seq, ncol, mask=mask, latlon=True, transform=transform_ll)
assert np.array_equal(new, old)
print("mask         latlon=True : bit-identical")

 int64  latlon=True: bit-identical
 int64  latlon=False: bit-identical


 int32  latlon=True: bit-identical
 int32  latlon=False: bit-identical
uint32  latlon=True: bit-identical
uint32  latlon=False: bit-identical
nextxy-jump  latlon=True : bit-identical


mask         latlon=True : bit-identical


In [5]:
def timeit(f, *args, repeat=3, **kwargs):
    """Return the fastest of `repeat` runs after two warm-up calls."""
    for _ in range(2):
        f(*args, **kwargs)  # warm-up / compile
    best = np.inf
    for _ in range(repeat):
        t0 = time.perf_counter()
        f(*args, **kwargs)
        best = min(best, time.perf_counter() - t0)
    return best

In [6]:
rows = []
for ncol, nrow, tag in [(500, 500, "0.25M"), (2000, 2000, "4M")]:
    idxs_ds, seq = synthetic_d8(ncol, nrow)
    rng = np.random.default_rng(0)
    seq_shuf = rng.permutation(seq)  # mimic the BFS walk order of idxs_seq
    for label, seq_ in [("seq", seq), ("shuffled", seq_shuf)]:
        t_old = timeit(stream_distance_real_old, idxs_ds, seq_, ncol, None, True, transform_ll)
        t_new = timeit(streams.stream_distance, idxs_ds, seq_, ncol, latlon=True, transform=transform_ll)
        rows.append((tag, label, "latlon", t_old, t_new))
        t_old = timeit(stream_distance_real_old, idxs_ds, seq_, ncol, None, False, transform_xy)
        t_new = timeit(streams.stream_distance, idxs_ds, seq_, ncol, latlon=False, transform=transform_xy)
        rows.append((tag, label, "projected", t_old, t_new))

print(f"{'size':>6} {'order':>8} {'crs':>9} {'old [ms]':>10} {'new [ms]':>10} {'speedup':>8}")
for tag, label, crs, t_old, t_new in rows:
    print(f"{tag:>6} {label:>8} {crs:>9} {t_old*1e3:>10.1f} {t_new*1e3:>10.1f} {t_old/t_new:>7.1f}x")

  size    order       crs   old [ms]   new [ms]  speedup
 0.25M      seq    latlon        6.7        0.9     7.6x
 0.25M      seq projected        1.9        0.7     2.6x
 0.25M shuffled    latlon        7.0        1.1     6.5x
 0.25M shuffled projected        2.2        0.9     2.3x
    4M      seq    latlon      109.3       14.1     7.7x
    4M      seq projected       30.5       11.9     2.6x
    4M shuffled    latlon      391.8      106.6     3.7x
    4M shuffled projected      167.9       76.0     2.2x


## Real basin: the built-in Rhine example

The repository ships a 682 x 997 cell D8 raster of the Rhine basin in WGS84
(`examples/rhine_d8.tif`, 0.01 deg). This runs in well under a second and checks
the new implementation against the previous one on real data.


In [7]:
import rasterio

with rasterio.open("rhine_d8.tif") as src:
    d8 = src.read(1)
    transform = np.asarray(src.transform)
    latlon = src.crs.is_geographic
flw = pyflwdir.from_array(d8, ftype="d8", transform=transform, latlon=latlon, cache=True)
print(f"shape {d8.shape}, index dtype {flw.idxs_ds.dtype}, valid cells {flw.nnodes}")

idxs_ds = flw.idxs_ds
seq = flw.idxs_seq
ncol = flw.shape[1]

t_old = timeit(stream_distance_real_old, idxs_ds, seq, ncol, None, True, transform)
t_new = timeit(streams.stream_distance, idxs_ds, seq, ncol, latlon=True, transform=transform)
old = stream_distance_real_old(idxs_ds, seq, ncol, None, True, transform)
new = streams.stream_distance(idxs_ds, seq, ncol, latlon=True, transform=transform)
assert np.array_equal(new, old)
print(f"old {t_old*1e3:.1f} ms   new {t_new*1e3:.1f} ms   speedup {t_old/t_new:.1f}x   bit-identical")

# public API
t0 = time.perf_counter()
ldn = flw.stream_distance(unit="m")
dt = time.perf_counter() - t0
assert np.array_equal(ldn.ravel(), new)
print(f"FlwdirRaster.stream_distance(unit='m'): {dt*1e3:.1f} ms, "
      f"max distance {ldn[ldn != -9999].max():.0f} m")

shape (682, 997), index dtype int32, valid cells 349847


old 11.9 ms   new 3.7 ms   speedup 3.2x   bit-identical
FlwdirRaster.stream_distance(unit='m'): 3.7 ms, max distance 1368155 m
